[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IfimoAI/moju/blob/main/examples/Notebooks/media/moju_slab_cooling_path_b.ipynb)

<h1 style="text-align: center;"> <b> Moju Demo — Physics Auditing for 1D Transient Slab Cooling
</h1>

#### **About this notebook**

This demo showcases Moju’s **bring-your-own-predictions** workflow using a 1D transient slab cooling problem. Your surrogate model (PINN, neural operator, CFD surrogate, etc.) remains fully user-defined — including the network architecture, collocation strategy, optimizer, and training loop.

#### **Moju’s role: Physics diagnostics and admissibility auditing**

Moju operates independently of training by auditing the predicted field variables (e.g. temperature T and its derivatives) against user-selected governing Laws, constitutive relationships, and optional dimensionless Groups. Users simply provide the model outputs together with the applicable coordinate dimension (1D, 2D, or 3D) and physics definitions.

#### **Why this matters**

Most Physics AI workflows primarily monitor governing equation residuals during training. Moju extends this by independently evaluating the implied constitutive behavior to detect hidden physical inconsistencies that may not appear in standard PDE losses. The goal is not to replace your training framework, but to provide a modular physics forensic layer for validating arbitrary surrogate models.

In [ ]:
# Colab / fresh environment
!pip install -q "moju==1.1.2" optax


#### Import relevant libraries

In [ ]:
import requests
import zipfile
import json
import jax.numpy as jnp

from io import BytesIO
from typing import Any
from google.colab import drive, files
from moju.piratio import Models, Laws
from moju.monitor import ResidualEngine, implied_group_specs_for_laws, visualize

### Load coordinates and predicted state variables from surrogate model  <br> <h5>(Model: PINN with 3 hidden layers consisting of 32, 64, and 32 Neurons sequentially; Input Layer has 2 neurons with non-dimensional x and t as inputs and dimensionsionless Temperature, Theta as output)</h5>

In [ ]:
# Bundled Path B demo state (wide2 eval grid)
url = "https://github.com/IfimoAI/moju/raw/main/examples/Notebooks/media/data/wide2_const_prop_1D_cooling_slab_test_state_pred.json.zip"

response = requests.get(url)
response.raise_for_status()

with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
    json_members = [
        name
        for name in zip_ref.namelist()
        if name.endswith(".json") and not name.startswith("__MACOSX")
    ]
    if not json_members:
        raise ValueError(f"No JSON member found in {url}")
    print("Files in zip:", json_members)
    with zip_ref.open(json_members[0]) as json_file:
        test_state_raw = json.load(json_file)


In [ ]:
def json_to_jax(obj: Any) -> Any:
    """ Helper function to convert saved state predictions from json format to jax"""
    if isinstance(obj, dict):
        return {k: json_to_jax(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return jnp.asarray(obj)
    if isinstance(obj, (int, float)):
        return jnp.asarray(obj)
    return obj

In [ ]:
# Convert raw training json state predictions to jax format
state_pred = json_to_jax(test_state_raw)

#### Define model Constants

In [ ]:
L = 0.1           # Domain length
k_solid = 200.0   # Reference thermal conductivity
rho_ref = 2700.0  # Reference density
cp = 900.0        # Specific heat capacity
h = 500.00       # Convection heat transfer coefficient

#### Define Moju forensic engine config

In [ ]:
constants = {"L": L, "cp": cp, "k": k_solid, "rho": rho_ref, "alpha": Models.thermal_diffusivity(k_solid, rho_ref, cp)}

In [ ]:
engine = ResidualEngine(constants=constants, laws=[{"name": "fourier_conduction", "state_map": {"T_t": "T_t", "T_laplacian": "T_xx", "fo": "fo", "t": "t", "L": "L"}}],
                              groups=implied_group_specs_for_laws(["fourier_conduction"]), best_effort_partial=False, default_coord_dimension=1)

#### Compute residuals from state predictions

In [ ]:
# Training residuals
engine.clear_log()
state_pred_residuals = engine.compute_residuals(state_pred, log_to_python=True)

#### Visualize Moju's Forensic Diagnostics

In [ ]:
# Visualize Moju Bring-Your-Own-Model (BYOM) training diagnostics
fig = visualize(engine.log, mode='eval', residuals=state_pred_residuals)
fig.show()

### **Key Observations**

#### The governing-law admissibility remained high (>99.99%) while constitutive admissibility fell below the acceptable threshold.<br>

#### This demonstrates that PDE satisfaction alone may not guarantee constitutive consistency.

#### The PINN model in this demo is an overparameterized model with the following architecture 2 x 128 x 128 x 128 x 1.

#### The model was trained on 64 x 48 = 3072 collocation points and the evaluation collocation points displayed in this demo is 8 x denser in time and space. hence, the evaluation has **196608** collocation points